In [0]:
from pyspark.sql import functions as F

CATALOG = "dbr_dev"
SCHEMA = "live_transit_monitor"

SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

#small agregation tables
GOLD_LIVE_KPI = f"{CATALOG}.{SCHEMA}.gold_live_kpi"
GOLD_ROUTE_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_route_summary"
GOLD_DELAY_DISTRIBUTION = f"{CATALOG}.{SCHEMA}.gold_delay_distribution"
GOLD_DESTINATION_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_destination_summary"

silver = spark.read.table(SILVER)

In [0]:
from pyspark.sql import functions as F, Window

gold_live_kpi = (
    silver.agg(
        F.countDistinct("vehicleId").alias("active_vehicles"),
        F.round(
            F.avg(F.when(F.col("has_trip"), F.col("delay_min"))),2
        ).alias("avg_delay_min"),
        F.round(
            F.max(F.when(F.col("has_trip"),F.col("delay_min"))),2
        ).alias("max_delay_min"),

        F.countDistinct(F.when( F.col("is_stopped"),F.col("vehicleId"))
        ).alias("stopped_vehicles"),

        F.countDistinct(
            F.when(F.col("is_delayed"),F.col("vehicleId"))
        ).alias("delayed_vehicles"),

        F.countDistinct(
            F.when(F.col("is_stopped") & (F.col("transportationType") == "Autobus"),F.col("vehicleId") )
        ).alias("stopped_buses"),

        F.countDistinct(
            F.when(F.col("is_stopped")& (F.col("transportationType") == "Tramwaj"), F.col("vehicleId") )
        ).alias("stopped_trams")
    )
)

display(gold_live_kpi)

In [0]:
(
    gold_live_kpi.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_LIVE_KPI)
)

In [0]:
#average delay, speed per route and top delayed routes
gold_route_summary = (
    silver.filter(F.col("routeId").isNotNull())
    .groupBy(
        "routeId",
        "routeShortName",
        "transportationType"
    )
    .agg(
        F.countDistinct("vehicleId").alias("active_vehicles"),
        F.round(
            F.avg(F.when(F.col("has_trip"),F.col("delay_min"))),2
        ).alias("avg_delay_min"),

        F.round(
            F.max(F.when(F.col("has_trip"),F.col("delay_min"))),2
        ).alias("max_delay_min"),

        F.round(
            F.avg(F.when( F.col("is_moving"), F.col("speed"))),2
        ).alias("avg_speed"),

        F.countDistinct(
            F.when( F.col("is_delayed"), F.col("vehicleId"))
        ).alias("delayed_vehicles"),

        F.countDistinct(
            F.when(F.col("is_stopped"),F.col("vehicleId") )
        ).alias("stopped_vehicles")
    )
)

display(gold_route_summary .orderBy(F.desc("avg_delay_min")))

In [0]:
(
    gold_route_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_ROUTE_SUMMARY)
)

In [0]:
#delay distribution
gold_delay_distribution = (
    silver
    .filter(F.col("has_trip"))
    .groupBy("delay_bucket")
    .agg(
        F.count("*").alias("records_count"),
        F.countDistinct("vehicleId").alias("vehicles_count")
    )
)

display(gold_delay_distribution)

In [0]:
(
    gold_delay_distribution.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DELAY_DISTRIBUTION)
)

In [0]:
gold_destination_summary = (
    silver.filter(
        F.col("has_trip") & F.col("headsign").isNotNull()
    )
    .groupBy(
        "headsign",
        "transportationType"
    )
    .agg(
        F.countDistinct("vehicleId").alias("active_vehicles"),
        F.round(
            F.avg("delay_min"),2).alias("avg_delay_min"),

        F.round(F.avg(F.when(F.col("is_moving"),F.col("speed"))),2
        ).alias("avg_speed"),

        F.countDistinct(
            F.when( F.col("is_delayed"),F.col("vehicleId"))
        ).alias("delayed_vehicles")
    )
)

display(
    gold_destination_summary
    .orderBy(F.desc("active_vehicles"))
)

In [0]:
(
    gold_destination_summary.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_DESTINATION_SUMMARY)
)